# Task 052: us-beamform-linarray-master — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Ultrasound beamforming using delay-and-sum with linear array

**Paper**: See repository documentation
**Repository**: https://github.com/csheaff/us-beamform-linarray

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 31.65 dB
- **SSIM**: N/A


### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import logging
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.interpolate import interp2d
import h5py
import time
import os

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Constants
N_TRANSMIT_BEAMS = 96
N_PROBE_CHANNELS = 32
TRANSMIT_FREQ = 1.5e6
TRANSMIT_FOCAL_DEPTH = 20e-3
SPEED_SOUND = 1540
ARRAY_PITCH = 2 * 1.8519e-4
SAMPLE_RATE = 27.72e6
TIME_OFFSET = 1.33e-6

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`arange2`, `get_tgc`, `preproc`, `beamform`, `beamform_df`, `envel_detect`, `log_compress`, `scan_convert`, `main`

In [ ]:
def arange2(start, stop=None, step=1):
    """Modified version of numpy.arange which corrects error associated with non-integer step size"""
    if stop is None:
        a = np.arange(start)
    else:
        a = np.arange(start, stop, step)
        if a[-1] > stop - step:
            a = np.delete(a, -1)
    return a

def get_tgc(alpha0, prop_dist):
    """Time-gain compensation"""
    n = 1  # approx. 1 for soft tissue
    alpha = alpha0 * (TRANSMIT_FREQ * 1e-6)**n
    tgc_gain = 10**(alpha * prop_dist * 100 / 20)
    return tgc_gain

def preproc(data, t, xd):
    """Preprocessing: TGC, filtering, interpolation, apodization"""
    fs = 1 / (t[1] - t[0])
    record_length = data.shape[2]
    a0 = 0.4

    # TGC
    zd = t * SPEED_SOUND / 2
    zd2 = zd**2
    dist1 = zd
    tgc = np.zeros((N_PROBE_CHANNELS, record_length))
    for r in range(N_PROBE_CHANNELS):
        dist2 = np.sqrt(xd[r]**2 + zd2)
        prop_dist = dist1 + dist2
        tgc[r, :] = get_tgc(a0, prop_dist)

    data_amp = np.zeros(data.shape)
    for m in range(N_TRANSMIT_BEAMS):
        data_amp[m, :, :] = data[m, :, :] * tgc

    # Filtering
    filt_ord = 201
    lc, hc = 0.5e6, 2.5e6
    lc = lc / (fs / 2)
    hc = hc / (fs / 2)
    B = signal.firwin(filt_ord, [lc, hc], pass_zero=False)

    # Interpolation
    interp_fact = 4
    fs_new = fs * interp_fact
    record_length2 = record_length * interp_fact
    
    # Apodization
    try:
        apod_win = signal.tukey(N_PROBE_CHANNELS)
    except AttributeError:
        # Fallback for some scipy versions or import structures
        try:
            from scipy.signal.windows import tukey
            apod_win = tukey(N_PROBE_CHANNELS)
        except ImportError:
            # Fallback to simple window if tukey is missing
            apod_win = np.hanning(N_PROBE_CHANNELS) # Close approximation or use ones
            # apod_win = np.ones(N_PROBE_CHANNELS)


    data_apod = np.zeros((N_TRANSMIT_BEAMS, N_PROBE_CHANNELS, record_length2))
    
    # Process
    for m in range(N_TRANSMIT_BEAMS):
        for n in range(N_PROBE_CHANNELS):
            w = data_amp[m, n, :]
            data_filt = signal.lfilter(B, 1, w)
            data_interp = signal.resample_poly(data_filt, interp_fact, 1)
            data_apod[m, n, :] = apod_win[n] * data_interp

    # New time vector
    freqs, delay = signal.group_delay((B, 1))
    delay = int(delay[0]) * interp_fact
    t2 = np.arange(record_length2) / fs_new + t[0] - delay / fs_new

    # Remove signal before t=0
    f = np.where(t2 < 0)[0]
    t2 = np.delete(t2, f)
    data_apod = data_apod[:, :, f[-1]+1:]

    return data_apod, t2

def beamform(data, t, xd, receive_focus):
    """Delay-and-sum beamforming with fixed focus"""
    Rf = receive_focus
    fs = 1 / (t[1] - t[0])
    delay_ind = np.zeros(N_PROBE_CHANNELS, dtype=int)
    for r in range(N_PROBE_CHANNELS):
        delay = Rf / SPEED_SOUND * (np.sqrt((xd[r] / Rf)**2 + 1) - 1)
        delay_ind[r] = int(round(delay * fs))
    max_delay = np.max(delay_ind)

    waveform_length = data.shape[2]
    image = np.zeros((N_TRANSMIT_BEAMS, waveform_length))
    for q in range(N_TRANSMIT_BEAMS):
        scan_line = np.zeros(waveform_length + max_delay)
        for r in range(N_PROBE_CHANNELS):
            delay_pad = np.zeros(delay_ind[r])
            fill_pad = np.zeros(len(scan_line) - waveform_length - delay_ind[r])
            waveform = data[q, r, :]
            # Fix concatenation shapes if needed, but original logic seems ok
            scan_line = scan_line + np.concatenate((fill_pad, waveform, delay_pad))
        image[q, :] = scan_line[max_delay:]
    return image

def beamform_df(data, t, xd):
    """Dynamic focusing beamforming"""
    fs = 1 / (t[2] - t[1])
    zd = t * SPEED_SOUND / 2
    zd2 = zd**2
    prop_dist = np.zeros((N_PROBE_CHANNELS, len(zd)))
    for r in range(N_PROBE_CHANNELS):
        dist1 = zd
        dist2 = np.sqrt(xd[r]**2 + zd2)
        prop_dist[r, :] = dist1 + dist2

    prop_dist_ind = np.round(prop_dist / SPEED_SOUND * fs).astype('int')
    
    # Handle out of bounds
    oob_inds = np.where(prop_dist_ind >= len(t))
    prop_dist_ind[oob_inds[0], oob_inds[1]] = len(t) - 1

    image = np.zeros((N_TRANSMIT_BEAMS, len(zd)))
    for q in range(N_TRANSMIT_BEAMS):
        data_received = data[q, ...]
        scan_line = np.zeros(len(zd))
        for r in range(N_PROBE_CHANNELS):
            v = data_received[r, :]
            scan_line = scan_line + v[prop_dist_ind[r, :]]
        image[q, :] = scan_line
    return image

def envel_detect(scan_line, t, method='hilbert'):
    """Envelope detection"""
    if method == 'hilbert':
        envelope = np.abs(signal.hilbert(scan_line))
    return envelope

def log_compress(data, dynamic_range, reject_level, bright_gain):
    """Log compression"""
    xd_b = 20 * np.log10(data + 1)
    xd_b2 = xd_b - np.max(xd_b)
    xd_b3 = xd_b2 + dynamic_range
    xd_b3[xd_b3 < 0] = 0
    xd_b3[xd_b3 <= reject_level] = 0
    xd_b3 = xd_b3 + bright_gain
    xd_b3[xd_b3 > dynamic_range] = dynamic_range
    return xd_b3

def scan_convert(data, xb, zb):
    """Scan conversion to image grid"""
    decim_fact = 8
    data = data[:, 0:-1:decim_fact]
    zb = zb[0:-1:decim_fact]

    interp_func = interp2d(zb, xb, data, kind='linear')
    dz = zb[1] - zb[0]
    xnew = arange2(xb[0], xb[-1] + dz, dz)
    znew = zb
    image_sC = interp_func(znew, xnew)
    return image_sC, znew, xnew

def main():
    data_path = 'example_us_bmode_sensor_data.h5'
    if not os.path.exists(data_path):
        logger.error(f"Data file not found at {data_path}")
        return

    logger.info("Loading data...")
    with h5py.File(data_path, 'r') as h5f:
        sensor_data = h5f['dataset_1'][:]

    logger.info(f"Data shape: {sensor_data.shape}")

    record_length = sensor_data.shape[2]
    time_vec = np.arange(record_length) / SAMPLE_RATE - TIME_OFFSET
    xd = np.arange(N_PROBE_CHANNELS) * ARRAY_PITCH
    xd = xd - np.max(xd) / 2

    logger.info("Preprocessing...")
    preproc_data, time_shifted = preproc(sensor_data, time_vec, xd)
    logger.info(f"Preprocessed shape: {preproc_data.shape}")

    # Reconstruction 1: Fixed Focus
    logger.info("Reconstruction 1: Fixed Focus Beamforming...")
    receive_focus = 30e-3
    image_bf = beamform(preproc_data, time_shifted, xd, receive_focus)

    # Reconstruction 2: Dynamic Focusing (High Quality)
    logger.info("Reconstruction 2: Dynamic Focusing Beamforming...")
    start_time = time.time()
    image_df = beamform_df(preproc_data, time_shifted, xd)
    logger.info(f"Dynamic focusing took {time.time() - start_time:.2f}s")

    # Post-processing
    logger.info("Post-processing (Envelope, Log Compress, Scan Convert)...")
    z = time_shifted * SPEED_SOUND / 2
    xd2 = np.arange(N_TRANSMIT_BEAMS) * ARRAY_PITCH
    xd2 = xd2 - np.max(xd2) / 2

    images_to_proc = {'Fixed Focus': image_bf, 'Dynamic Focus': image_df}
    processed_images = {}
    
    # Common axes for scan conversion
    x_sc_common = None
    z_sc_common = None

    for name, im in images_to_proc.items():
        # Truncate near field
        f = np.where(z < 5e-3)[0]
        z_trunc = np.delete(z, f)
        im_trunc = im[:, f[-1]+1:]

        # Envelope
        for m in range(N_TRANSMIT_BEAMS):
            im_trunc[m, :] = envel_detect(im_trunc[m, :], 2*z_trunc/SPEED_SOUND)

        # Log Compress
        DR = 35
        image_log = log_compress(im_trunc, DR, 0, 0)

        # Scan Convert
        image_sc, z_sc, x_sc = scan_convert(image_log, xd2, z_trunc)
        
        # Normalize to 0-255
        image_sc_norm = np.round(255 * image_sc / DR)
        image_sc_norm[image_sc_norm < 0] = 0
        image_sc_norm[image_sc_norm > 255] = 255
        image_final = image_sc_norm.astype('uint8').T # Transpose for display
        
        processed_images[name] = image_final
        if x_sc_common is None:
            x_sc_common = x_sc
            z_sc_common = z_sc

    # Evaluation
    # Using Dynamic Focus as reference
    ref = processed_images['Dynamic Focus']
    target = processed_images['Fixed Focus']
    
    psnr_val, rmse_val = calculate_metrics(target, ref)
    print(f"\nEvaluation Results (Fixed Focus vs Dynamic Focus Reference):")
    print(f"PSNR: {psnr_val:.2f} dB")
    print(f"RMSE: {rmse_val:.2f}")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    ax = axes[0]
    ax.imshow(target, extent=[x_sc_common[0]*1e3, x_sc_common[-1]*1e3, z_sc_common[-1]*1e3, z_sc_common[0]*1e3], cmap='gray')
    ax.set_title(f"Fixed Focus (PSNR: {psnr_val:.2f} dB)")
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("z (mm)")
    
    ax = axes[1]
    ax.imshow(ref, extent=[x_sc_common[0]*1e3, x_sc_common[-1]*1e3, z_sc_common[-1]*1e3, z_sc_common[0]*1e3], cmap='gray')
    ax.set_title("Dynamic Focus (Reference)")
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("z (mm)")
    
    plt.tight_layout()
    plt.savefig('reconstruction_comparison.png')
    logger.info("Result saved to reconstruction_comparison.png")

## 4. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `calculate_metrics`

In [ ]:
def calculate_metrics(img_pred, img_ref):
    """Calculate PSNR and RMSE between two images"""
    # Ensure they are same size
    if img_pred.shape != img_ref.shape:
        # Resize pred to ref
        # Using simple interpolation or just cropping if close
        # For this script, let's assume they are scan converted to same grid
        pass
    
    mse = np.mean((img_pred - img_ref) ** 2)
    if mse == 0:
        return float('inf'), 0
    
    data_range = np.max([np.max(img_pred), np.max(img_ref)]) - np.min([np.min(img_pred), np.min(img_ref)])
    psnr = 20 * np.log10(data_range / np.sqrt(mse))
    rmse = np.sqrt(mse)
    
    return psnr, rmse

## 5. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
data_path = 'example_us_bmode_sensor_data.h5'
if not os.path.exists(data_path):
    logger.error(f"Data file not found at {data_path}")
    return

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
logger.info("Loading data...")
with h5py.File(data_path, 'r') as h5f:
    sensor_data = h5f['dataset_1'][:]

logger.info(f"Data shape: {sensor_data.shape}")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
record_length = sensor_data.shape[2]
time_vec = np.arange(record_length) / SAMPLE_RATE - TIME_OFFSET
xd = np.arange(N_PROBE_CHANNELS) * ARRAY_PITCH
xd = xd - np.max(xd) / 2

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
logger.info("Preprocessing...")
preproc_data, time_shifted = preproc(sensor_data, time_vec, xd)
logger.info(f"Preprocessed shape: {preproc_data.shape}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Reconstruction 1: Fixed Focus
logger.info("Reconstruction 1: Fixed Focus Beamforming...")
receive_focus = 30e-3
image_bf = beamform(preproc_data, time_shifted, xd, receive_focus)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Reconstruction 2: Dynamic Focusing (High Quality)
logger.info("Reconstruction 2: Dynamic Focusing Beamforming...")
start_time = time.time()
image_df = beamform_df(preproc_data, time_shifted, xd)
logger.info(f"Dynamic focusing took {time.time() - start_time:.2f}s")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Post-processing
logger.info("Post-processing (Envelope, Log Compress, Scan Convert)...")
z = time_shifted * SPEED_SOUND / 2
xd2 = np.arange(N_TRANSMIT_BEAMS) * ARRAY_PITCH
xd2 = xd2 - np.max(xd2) / 2

images_to_proc = {'Fixed Focus': image_bf, 'Dynamic Focus': image_df}
processed_images = {}

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Common axes for scan conversion
x_sc_common = None
z_sc_common = None

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
for name, im in images_to_proc.items():
    # Truncate near field
    f = np.where(z < 5e-3)[0]
    z_trunc = np.delete(z, f)
    im_trunc = im[:, f[-1]+1:]

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Envelope
    for m in range(N_TRANSMIT_BEAMS):
        im_trunc[m, :] = envel_detect(im_trunc[m, :], 2*z_trunc/SPEED_SOUND)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Log Compress
    DR = 35
    image_log = log_compress(im_trunc, DR, 0, 0)

# Scan Convert
    image_sc, z_sc, x_sc = scan_convert(image_log, xd2, z_trunc)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Normalize to 0-255
    image_sc_norm = np.round(255 * image_sc / DR)
    image_sc_norm[image_sc_norm < 0] = 0
    image_sc_norm[image_sc_norm > 255] = 255
    image_final = image_sc_norm.astype('uint8').T # Transpose for display

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
processed_images[name] = image_final
    if x_sc_common is None:
        x_sc_common = x_sc
        z_sc_common = z_sc

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Evaluation
# Using Dynamic Focus as reference
ref = processed_images['Dynamic Focus']
target = processed_images['Fixed Focus']

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
psnr_val, rmse_val = calculate_metrics(target, ref)
print(f"\nEvaluation Results (Fixed Focus vs Dynamic Focus Reference):")
print(f"PSNR: {psnr_val:.2f} dB")
print(f"RMSE: {rmse_val:.2f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
ax = axes[0]
ax.imshow(target, extent=[x_sc_common[0]*1e3, x_sc_common[-1]*1e3, z_sc_common[-1]*1e3, z_sc_common[0]*1e3], cmap='gray')
ax.set_title(f"Fixed Focus (PSNR: {psnr_val:.2f} dB)")
ax.set_xlabel("x (mm)")
ax.set_ylabel("z (mm)")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
ax = axes[1]
ax.imshow(ref, extent=[x_sc_common[0]*1e3, x_sc_common[-1]*1e3, z_sc_common[-1]*1e3, z_sc_common[0]*1e3], cmap='gray')
ax.set_title("Dynamic Focus (Reference)")
ax.set_xlabel("x (mm)")
ax.set_ylabel("z (mm)")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
plt.tight_layout()
plt.savefig('reconstruction_comparison.png')
logger.info("Result saved to reconstruction_comparison.png")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 6. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **us-beamform-linarray-master**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=31.65 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: See documentation
- Repository: https://github.com/csheaff/us-beamform-linarray
